# Imports

In [ ]:
import numpy as np # A useful package for dealing with mathematical processes, we will be using it this week for vectors and matrices
import pandas as pd # A common package for viewing tabular data
import sklearn.linear_model, sklearn.datasets # We want to be able to access the sklearn datasets again, also we are using some model evaluation
from sklearn.preprocessing import StandardScaler, MinMaxScaler # We will be using the imbuilt sclaing functions sklearn provides
import matplotlib.pyplot as plt # We will be using Matplotlib for our graphs
from sklearn.preprocessing import PolynomialFeatures # A preprocessing function allowing us to include polynomial features into our model
from google.colab import files # We will be importing a csv file I have provided for one section.
from sklearn.preprocessing import LabelEncoder, OneHotEncoder # We will be using these to encode categorical features
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import seaborn as sns; sns.set()
from sklearn.metrics import  accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, mean_squared_error, r2_score , confusion_matrix, ConfusionMatrixDisplay # required for evaluating classification models
import joblib
from google.colab import files


# Loading Datasets


In [ ]:
uploaded = files.upload()
front_squat_angle_data = pd.read_csv('front.csv')
side_squat_angle_data = pd.read_csv('side.csv')

Saving side.csv to side (1).csv
Saving front.csv to front (1).csv


# Data Processing


In [ ]:
# load both CSVs into dataframes
front_df = pd.read_csv('front.csv')
side_df = pd.read_csv('side.csv')

# check the shape and first few rows of each dataset
print("Front CSV shape:", front_df.shape)
display(front_df.head())

print("Side CSV shape:", side_df.shape)
display(side_df.head())

# check how many rows per label
print("\nFront label distribution:")
display(front_df['class'].value_counts())

print("\nSide label distribution:")
display(side_df['class'].value_counts())

Front CSV shape: (4060, 13)


,file,class,left_knee_angle,left_hip_angle,left_trunk_angle,right_knee_angle,right_hip_angle,right_trunk_angle,knee_distance,ankle_distance,knee_ankle_ratio,left_knee_foot_offset,right_knee_foot_offset
0,squat_front_1.mp4,good,178.08,172.21,173.15,176.78,172.25,173.74,0.076234,0.083664,0.911195,0.006732,0.000698
1,squat_front_1.mp4,good,177.90,172.08,173.10,176.84,172.25,173.71,0.076564,0.083840,0.913216,0.006412,0.000864
2,squat_front_1.mp4,good,177.70,171.94,173.04,176.85,172.24,173.71,0.076887,0.083934,0.916047,0.006111,0.000935
3,squat_front_1.mp4,good,177.60,171.86,173.01,176.87,172.25,173.70,0.077051,0.084042,0.916810,0.005969,0.001022
4,squat_front_1.mp4,good,177.59,171.84,172.99,176.89,172.24,173.68,0.077100,0.084119,0.916554,0.005947,0.001072


Side CSV shape: (4044, 13)


,file,class,left_knee_angle,left_hip_angle,left_trunk_angle,right_knee_angle,right_hip_angle,right_trunk_angle,knee_distance,ankle_distance,knee_ankle_ratio,left_knee_foot_offset,right_knee_foot_offset
0,squat_side_1.mp4,good,179.33,177.26,177.58,178.55,178.10,178.79,0.034627,0.034307,1.009335,0.011281,0.011601
1,squat_side_1.mp4,good,179.63,176.66,176.84,178.73,178.22,178.83,0.034619,0.035840,0.965957,0.012300,0.011080
2,squat_side_1.mp4,good,179.81,176.25,176.34,178.83,178.36,178.92,0.034727,0.037004,0.938460,0.013047,0.010770
3,squat_side_1.mp4,good,179.91,176.20,176.25,179.41,178.80,179.08,0.033869,0.037637,0.899875,0.013616,0.009848
4,squat_side_1.mp4,good,179.91,176.23,176.27,179.46,178.99,179.25,0.033767,0.037791,0.893519,0.013709,0.009685



Front label distribution:


,count
class,
good,2254
knees_in,1806



Side label distribution:


,count
class,
good,2065
leaning_forward,1979


# Front Model


In [ ]:
# separate features and labels for front view
# drop file and class columns as they are not features
X_front = front_squat_angle_data.drop(columns=['file', 'class'])
y_front = front_squat_angle_data['class']

# encode the labels into numbers (good = 0, knees_in = 1)
le_front = LabelEncoder()
y_front = le_front.fit_transform(y_front)

# split into 80% training and 20% testing
X_front_train, X_front_test, y_front_train, y_front_test = train_test_split(
    X_front, y_front, test_size=0.2, random_state=42
)

# scale the features so all angles are on the same scale
scaler_front = MinMaxScaler()
X_front_train = scaler_front.fit_transform(X_front_train)
X_front_test  = scaler_front.transform(X_front_test)

print("Front view data ready for training!")
print(f"Training samples : {len(X_front_train)}")
print(f"Testing samples  : {len(X_front_test)}")
print(f"Classes          : {le_front.classes_}")

Front view data ready for training!
Training samples : 3248
Testing samples  : 812
Classes          : ['good' 'knees_in']


In [ ]:
# function to evaluate a model on both training and test data
# shows train vs test metrics to help spot overfitting
def evaluate_model_front(model, model_name="Model"):

    # predict on both train and test sets
    y_pred_train = model.predict(X_front_train)
    y_pred_test  = model.predict(X_front_test)

    # calculate metrics for both sets
    results_df = pd.DataFrame([
        ['Accuracy',  accuracy_score(y_front_train, y_pred_train),  accuracy_score(y_front_test, y_pred_test)],
        ['Precision', precision_score(y_front_train, y_pred_train), precision_score(y_front_test, y_pred_test)],
        ['Recall',    recall_score(y_front_train, y_pred_train),    recall_score(y_front_test, y_pred_test)],
        ['F1 Score',  f1_score(y_front_train, y_pred_train),        f1_score(y_front_test, y_pred_test)]
    ], columns=['Metric', 'Train', 'Test'])

    print(f"\n{model_name} Results:")
    display(results_df)

    return results_df, y_pred_train, y_pred_test

# define all models to be trained and compared
models_front = {
    "Logistic Regression" : LogisticRegression(max_iter=1000, random_state=42),
    "Ridge Classifier"    : RidgeClassifier(),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting"   : GradientBoostingClassifier(random_state=42),
    "SVC"                 : SVC(kernel='rbf', random_state=42),
    "KNN"                 : KNeighborsClassifier(n_neighbors=5),
    "Decision Tree"       : DecisionTreeClassifier(random_state=42),
    "Gaussian NB"         : GaussianNB(),
    "XGBoost"             : XGBClassifier(random_state=42, eval_metric='logloss')
}

# train each model and evaluate it
for model_name, model in models_front.items():
    model.fit(X_front_train, y_front_train)
    evaluate_model_front(model, model_name)


Logistic Regression Results:


,Metric,Train,Test
0,Accuracy,0.791256,0.786946
1,Precision,0.774035,0.745763
2,Recall,0.755997,0.760807
3,F1 Score,0.764910,0.753210



Ridge Classifier Results:


,Metric,Train,Test
0,Accuracy,0.797106,0.793103
1,Precision,0.793255,0.757925
2,Recall,0.741604,0.757925
3,F1 Score,0.766560,0.757925



Random Forest Results:


...

,Metric,Train,Test
0,Accuracy,1.0,0.948276
1,Precision,1.0,0.936963
2,Recall,1.0,0.942363
3,F1 Score,1.0,0.939655



Gaussian NB Results:


,Metric,Train,Test
0,Accuracy,0.665333,0.660099
1,Precision,0.589080,0.565863
2,Recall,0.843043,0.878963
3,F1 Score,0.693544,0.688488



XGBoost Results:


,Metric,Train,Test
0,Accuracy,1.0,0.990148
1,Precision,1.0,0.994169
2,Recall,1.0,0.982709
3,F1 Score,1.0,0.988406


# Hyperparamater tunning for Front Squat Model optimisation

## KNN

In [ ]:
# ---- KNN Tuning - Front View ----
knn_param_grid = [
    {
        'n_neighbors' : [3, 5, 7, 9, 11],
        'weights'     : ['uniform', 'distance'],
        'metric'      : ['minkowski', 'manhattan'],
        'p'           : [1, 2],
        'algorithm'   : ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'leaf_size'   : [20, 30, 40, 50]
    }
]
knn_front = KNeighborsClassifier()
knn_tuning = RandomizedSearchCV(
    knn_front, knn_param_grid,
    n_iter=50, cv=5, scoring='f1',
    n_jobs=-1, verbose=1, random_state=42
)
knn_tuning.fit(X_front_train, y_front_train)
knn_best_params_front = knn_tuning.best_params_
print("KNN best parameters:", knn_best_params_front)
knn_best_front = KNeighborsClassifier(**knn_best_params_front)
knn_best_front.fit(X_front_train, y_front_train)
evaluate_model_front(knn_best_front, "KNN Optimised")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
KNN best parameters: {'weights': 'distance', 'p': 2, 'n_neighbors': 3, 'metric': 'minkowski', 'leaf_size': 30, 'algorithm': 'brute'}

KNN Optimised Results:


,Metric,Train,Test
0,Accuracy,1.0,0.977833
1,Precision,1.0,0.991045
2,Recall,1.0,0.956772
3,F1 Score,1.0,0.973607


(      Metric  Train      Test
 0   Accuracy    1.0  0.977833
 1  Precision    1.0  0.991045
 2     Recall    1.0  0.956772
 3   F1 Score    1.0  0.973607,
 array([0, 1, 0, ..., 0, 1, 1]),
 array([1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1,
        0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1,
        1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0,
        0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0,
        0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1,
        0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0,
        1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0,
        1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 

## Random Forest Tree

In [ ]:
# ---- Random Forest Tuning - Front View ----
rf_param_grid = {
    'n_estimators'      : [100, 200, 500],
    'max_depth'         : [10, 20, 50, None],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf'  : [1, 2, 4],
    'max_features'      : ['sqrt', 'log2', None],
    'class_weight'      : ['balanced', None]
}
rf_front = RandomForestClassifier(random_state=42)
rf_tuning = RandomizedSearchCV(
    rf_front, rf_param_grid,
    n_iter=50, cv=5, scoring='f1',
    n_jobs=-1, verbose=1, random_state=42
)
rf_tuning.fit(X_front_train, y_front_train)
rf_best_params_front = rf_tuning.best_params_
print("Random Forest best parameters:", rf_best_params_front)
rf_best_front = RandomForestClassifier(**rf_best_params_front, random_state=42)
rf_best_front.fit(X_front_train, y_front_train)
evaluate_model_front(rf_best_front, "Random Forest Optimised")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Random Forest best parameters: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20, 'class_weight': 'balanced'}

Random Forest Optimised Results:


,Metric,Train,Test
0,Accuracy,1.0,0.982759
1,Precision,1.0,0.994065
2,Recall,1.0,0.965418
3,F1 Score,1.0,0.979532


(      Metric  Train      Test
 0   Accuracy    1.0  0.982759
 1  Precision    1.0  0.994065
 2     Recall    1.0  0.965418
 3   F1 Score    1.0  0.979532,
 array([0, 1, 0, ..., 0, 1, 1]),
 array([1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1,
        0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1,
        1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0,
        0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0,
        0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1,
        0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0,
        1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0,
        1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 

## XGB

In [ ]:
# ---- XGBoost Tuning - Front View ----
xgb_param_grid = {
    'n_estimators'    : [100, 300, 500],
    'max_depth'       : [3, 6, 10],
    'learning_rate'   : [0.01, 0.1, 0.2],
    'subsample'       : [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma'           : [0, 1, 5],
    'reg_alpha'       : [0, 0.1, 1],
    'reg_lambda'      : [0.1, 1, 10],
    'scale_pos_weight': [1, 10, 20]
}
xgb_front = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_tuning = RandomizedSearchCV(
    xgb_front, xgb_param_grid,
    n_iter=50, cv=5, scoring='f1',
    n_jobs=-1, verbose=1, random_state=42
)
xgb_tuning.fit(X_front_train, y_front_train)
xgb_best_params_front = xgb_tuning.best_params_
print("XGBoost best parameters:", xgb_best_params_front)
xgb_best_front = XGBClassifier(**xgb_best_params_front, eval_metric='logloss', random_state=42)
xgb_best_front.fit(X_front_train, y_front_train)
evaluate_model_front(xgb_best_front, "XGBoost Optimised")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
XGBoost best parameters: {'subsample': 0.6, 'scale_pos_weight': 20, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 1.0}

XGBoost Optimised Results:


,Metric,Train,Test
0,Accuracy,1.0,0.991379
1,Precision,1.0,0.988506
2,Recall,1.0,0.991354
3,F1 Score,1.0,0.989928


(      Metric  Train      Test
 0   Accuracy    1.0  0.991379
 1  Precision    1.0  0.988506
 2     Recall    1.0  0.991354
 3   F1 Score    1.0  0.989928,
 array([0, 1, 0, ..., 0, 1, 1]),
 array([1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1,
        0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1,
        1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0,
        0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0,
        0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1,
        0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0,
        1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0,
        1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 

## Comparing

In [ ]:
print("\n---- Optimised Model Comparison - Front View ----")
optimised_results_front = {}

for name, model in [("XGBoost", xgb_best_front), ("Random Forest", rf_best_front), ("KNN", knn_best_front)]:
    y_pred = model.predict(X_front_test)
    optimised_results_front[name] = {
        "accuracy"  : round(accuracy_score(y_front_test, y_pred), 4),
        "precision" : round(precision_score(y_front_test, y_pred), 4),
        "recall"    : round(recall_score(y_front_test, y_pred), 4),
        "f1"        : round(f1_score(y_front_test, y_pred), 4)
    }

display(pd.DataFrame(optimised_results_front).T.sort_values('f1', ascending=False))
print("\nFront view tuning has been completed")


---- Optimised Model Comparison - Front View ----


,accuracy,precision,recall,f1
XGBoost,0.9914,0.9885,0.9914,0.9899
Random Forest,0.9828,0.9941,0.9654,0.9795
KNN,0.9778,0.9910,0.9568,0.9736



Front view tuning has been completed


# Exporting Front Model and Scaler

## front model to a pkl file

In [ ]:
joblib.dump(xgb_best_front, 'front_model.pkl')
files.download('front_model.pkl')
print("front_model.pkl saved!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

front_model.pkl saved!


## scaler

In [ ]:
joblib.dump(le_front, 'front_label_encoder.pkl')
files.download('front_scaler.pkl')
print("front_scaler.pkl saved!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

front_scaler.pkl saved!


## encoder

In [ ]:
joblib.dump(le_front, 'front_label_encoder.pkl')
files.download('front_label_encoder.pkl')
print("front_label_encoder.pkl saved!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

front_label_encoder.pkl saved!


# Side Model

## Data Processing

In [ ]:
# separate features and labels for side view
X_side = side_squat_angle_data.drop(columns=['file', 'class'])
y_side = side_squat_angle_data['class']

# encode the labels into numbers (good = 0, leaning_forward = 1)
le_side = LabelEncoder()
y_side = le_side.fit_transform(y_side)

# split into 80% training and 20% testing
X_side_train, X_side_test, y_side_train, y_side_test = train_test_split(
    X_side, y_side, test_size=0.2, random_state=42
)

# scale the features so all angles are on the same scale
scaler_side = MinMaxScaler()
X_side_train = scaler_side.fit_transform(X_side_train)
X_side_test  = scaler_side.transform(X_side_test)

print("Side view data ready for training!")
print(f"Training samples : {len(X_side_train)}")
print(f"Testing samples  : {len(X_side_test)}")
print(f"Classes          : {le_side.classes_}")

Side view data ready for training!
Training samples : 3235
Testing samples  : 809
Classes          : ['good' 'leaning_forward']


In [ ]:
# function to evaluate a model on both training and test data
# shows train vs test metrics to help spot overfitting
def evaluate_model_side(model, model_name="Model"):

    # predict on both train and test sets
    y_pred_train = model.predict(X_side_train)
    y_pred_test  = model.predict(X_side_test)

    # calculate metrics for both sets
    results_df = pd.DataFrame([
        ['Accuracy',  accuracy_score(y_side_train, y_pred_train),  accuracy_score(y_side_test, y_pred_test)],
        ['Precision', precision_score(y_side_train, y_pred_train), precision_score(y_side_test, y_pred_test)],
        ['Recall',    recall_score(y_side_train, y_pred_train),    recall_score(y_side_test, y_pred_test)],
        ['F1 Score',  f1_score(y_side_train, y_pred_train),        f1_score(y_side_test, y_pred_test)]
    ], columns=['Metric', 'Train', 'Test'])

    print(f"\n{model_name} Results:")
    display(results_df)

    return results_df, y_pred_train, y_pred_test

# define all models to be trained and compared
models_side = {
    "Logistic Regression" : LogisticRegression(max_iter=1000, random_state=42),
    "Ridge Classifier"    : RidgeClassifier(),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting"   : GradientBoostingClassifier(random_state=42),
    "SVC"                 : SVC(kernel='rbf', random_state=42),
    "KNN"                 : KNeighborsClassifier(n_neighbors=5),
    "Decision Tree"       : DecisionTreeClassifier(random_state=42),
    "Gaussian NB"         : GaussianNB(),
    "XGBoost"             : XGBClassifier(random_state=42, eval_metric='logloss')
}

# train each model and evaluate it
for model_name, model in models_side.items():
    model.fit(X_side_train, y_side_train)
    evaluate_model_side(model, model_name)


Logistic Regression Results:


,Metric,Train,Test
0,Accuracy,0.715611,0.724351
1,Precision,0.719794,0.693878
2,Recall,0.698254,0.725333
3,F1 Score,0.708861,0.709257



Ridge Classifier Results:


,Metric,Train,Test
0,Accuracy,0.727975,0.734240
1,Precision,0.734153,0.704082
2,Recall,0.707606,0.736000
3,F1 Score,0.720635,0.719687



Random Forest Results:


...

,Metric,Train,Test
0,Accuracy,1.0,0.866502
1,Precision,1.0,0.831266
2,Recall,1.0,0.893333
3,F1 Score,1.0,0.861183



Gaussian NB Results:


,Metric,Train,Test
0,Accuracy,0.562597,0.603214
1,Precision,0.583260,0.591216
2,Recall,0.412718,0.466667
3,F1 Score,0.483388,0.521610



XGBoost Results:


,Metric,Train,Test
0,Accuracy,1.0,0.919654
1,Precision,1.0,0.889447
2,Recall,1.0,0.944000
3,F1 Score,1.0,0.915912


# Hyperparamater tunning for Side Squat Model optimisation

## XGB

In [ ]:
xgb_param_grid_side = {
    'n_estimators'    : [100, 300, 500],
    'max_depth'       : [3, 6, 10],
    'learning_rate'   : [0.01, 0.1, 0.2],
    'subsample'       : [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma'           : [0, 1, 5],
    'reg_alpha'       : [0, 0.1, 1],
    'reg_lambda'      : [0.1, 1, 10],
    'scale_pos_weight': [1, 10, 20]
}

xgb_side = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_tuning_side = RandomizedSearchCV(
    xgb_side, xgb_param_grid_side,
    n_iter=50, cv=5, scoring='f1',
    n_jobs=-1, verbose=1, random_state=42
)
xgb_tuning_side.fit(X_side_train, y_side_train)

xgb_best_params_side = xgb_tuning_side.best_params_
print("XGBoost best parameters:", xgb_best_params_side)

xgb_best_side = XGBClassifier(**xgb_best_params_side, eval_metric='logloss', random_state=42)
xgb_best_side.fit(X_side_train, y_side_train)
evaluate_model_side(xgb_best_side, "XGBoost Optimised")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
XGBoost best parameters: {'subsample': 0.6, 'scale_pos_weight': 20, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 1.0}

XGBoost Optimised Results:


,Metric,Train,Test
0,Accuracy,1.0,0.904821
1,Precision,1.0,0.849765
2,Recall,1.0,0.965333
3,F1 Score,1.0,0.903870


(      Metric  Train      Test
 0   Accuracy    1.0  0.904821
 1  Precision    1.0  0.849765
 2     Recall    1.0  0.965333
 3   F1 Score    1.0  0.903870,
 array([1, 0, 1, ..., 0, 1, 1]),
 array([1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1,
        1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1,
        0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0,
        1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0,
        0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0,
        1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0,
        0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0,
        0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1,
        0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 

## Random Forest Tree

In [ ]:
rf_param_grid_side = {
    'n_estimators'      : [100, 200, 500],
    'max_depth'         : [10, 20, 50, None],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf'  : [1, 2, 4],
    'max_features'      : ['sqrt', 'log2', None],
    'class_weight'      : ['balanced', None]
}

rf_side = RandomForestClassifier(random_state=42)
rf_tuning_side = RandomizedSearchCV(
    rf_side, rf_param_grid_side,
    n_iter=50, cv=5, scoring='f1',
    n_jobs=-1, verbose=1, random_state=42
)
rf_tuning_side.fit(X_side_train, y_side_train)

rf_best_params_side = rf_tuning_side.best_params_
print("Random Forest best parameters:", rf_best_params_side)

rf_best_side = RandomForestClassifier(**rf_best_params_side, random_state=42)
rf_best_side.fit(X_side_train, y_side_train)
evaluate_model_side(rf_best_side, "Random Forest Optimised")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Random Forest best parameters: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20, 'class_weight': 'balanced'}

Random Forest Optimised Results:


,Metric,Train,Test
0,Accuracy,0.999073,0.914710
1,Precision,0.998133,0.875000
2,Recall,1.000000,0.952000
3,F1 Score,0.999066,0.911877


(      Metric     Train      Test
 0   Accuracy  0.999073  0.914710
 1  Precision  0.998133  0.875000
 2     Recall  1.000000  0.952000
 3   F1 Score  0.999066  0.911877,
 array([1, 0, 1, ..., 0, 1, 1]),
 array([1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1,
        1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1,
        0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0,
        1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0,
        0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0,
        1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
        0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0,
        0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1,
        0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 

## KNN



In [ ]:
knn_param_grid_side = [
    {
        'n_neighbors' : [3, 5, 7, 9, 11],
        'weights'     : ['uniform', 'distance'],
        'metric'      : ['minkowski', 'manhattan'],
        'p'           : [1, 2],
        'algorithm'   : ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'leaf_size'   : [20, 30, 40, 50]
    }
]

knn_side = KNeighborsClassifier()
knn_tuning_side = RandomizedSearchCV(
    knn_side, knn_param_grid_side,
    n_iter=50, cv=5, scoring='f1',
    n_jobs=-1, verbose=1, random_state=42
)
knn_tuning_side.fit(X_side_train, y_side_train)

knn_best_params_side = knn_tuning_side.best_params_
print("KNN best parameters:", knn_best_params_side)

knn_best_side = KNeighborsClassifier(**knn_best_params_side)
knn_best_side.fit(X_side_train, y_side_train)
evaluate_model_side(knn_best_side, "KNN Optimised")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
KNN best parameters: {'weights': 'distance', 'p': 2, 'n_neighbors': 3, 'metric': 'minkowski', 'leaf_size': 30, 'algorithm': 'brute'}

KNN Optimised Results:


,Metric,Train,Test
0,Accuracy,1.0,0.938195
1,Precision,1.0,0.905237
2,Recall,1.0,0.968000
3,F1 Score,1.0,0.935567


(      Metric  Train      Test
 0   Accuracy    1.0  0.938195
 1  Precision    1.0  0.905237
 2     Recall    1.0  0.968000
 3   F1 Score    1.0  0.935567,
 array([1, 0, 1, ..., 0, 1, 1]),
 array([1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1,
        1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1,
        0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0,
        1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0,
        0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
        0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0,
        0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1,
        0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 

## Comparing

In [ ]:
print("\n---- Optimised Model Comparison - Side View ----")
optimised_results_side = {}

for name, model in [("XGBoost", xgb_best_side), ("Random Forest", rf_best_side), ("KNN", knn_best_side)]:
    y_pred = model.predict(X_side_test)
    optimised_results_side[name] = {
        "accuracy"  : round(accuracy_score(y_side_test, y_pred), 4),
        "precision" : round(precision_score(y_side_test, y_pred), 4),
        "recall"    : round(recall_score(y_side_test, y_pred), 4),
        "f1"        : round(f1_score(y_side_test, y_pred), 4)
    }

display(pd.DataFrame(optimised_results_side).T.sort_values('f1', ascending=False))
print("\nSide view tuning complete")


---- Optimised Model Comparison - Side View ----


,accuracy,precision,recall,f1
KNN,0.9382,0.9052,0.9680,0.9356
Random Forest,0.9147,0.8750,0.9520,0.9119
XGBoost,0.9048,0.8498,0.9653,0.9039



Side view tuning complete


# Exporting Side Model and Scaler

## side model to a pkl

In [ ]:
joblib.dump(knn_best_side, 'side_model.pkl')
files.download('side_model.pkl')
print("side_model.pkl saved!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

side_model.pkl saved!


## scaler

In [ ]:
joblib.dump(scaler_side, 'side_scaler.pkl')
files.download('side_scaler.pkl')
print("side_scaler.pkl saved!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

side_scaler.pkl saved!


## encoder

In [ ]:
joblib.dump(le_side, 'side_label_encoder.pkl')
files.download('side_label_encoder.pkl')
print("side_label_encoder.pkl saved!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

side_label_encoder.pkl saved!
